# Guide Ranker Training

Fine-tunes [Qwen/Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B) as a 12-class classifier
to predict `iterations_to_reach` (0–10, or 11 = unreached) from (goal, guide) s-expression pairs.

Lower predicted class = better guide.

Run with: `uv run jupyter notebook ranker.ipynb`

In [ ]:
import math
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.metrics import (
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    fbeta_score,
    matthews_corrcoef,
)
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3.5-2B"
MAX_LENGTH = 256  # max tokens per (goal, guide) pair
NUM_CLASSES = 12  # 0–10 = iterations_to_reach, 11 = unreached
SAMPLE_N = 10_000  # number of training+val examples to subsample
VAL_FRAC = 0.1  # fraction of data for validation
BASE_BATCH_SIZE = 16  # reference batch size for LR scaling
BASE_LR = 2e-5  # LR at reference batch size
BATCH_SIZE = 128
LR = BASE_LR * math.sqrt(BATCH_SIZE / BASE_BATCH_SIZE)  # sqrt scaling
EPOCHS = 3
SEED = 42

CLASS_NAMES = [str(i) for i in range(11)] + ["unreached"]

# ── Output directory (everything in one place) ────────────────────────
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_DIR = Path(f"runs/ranker-{timestamp}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = RUN_DIR / "log.txt"


def log(msg, obj=None):
    """Display and append to log file. If obj is given, display it too."""
    try:
        display(msg)
    except NameError:
        print(msg)
    with open(LOG_FILE, "a") as f:
        f.write(str(msg) + "\n")
        if obj is not None:
            f.write(str(obj) + "\n")
    if obj is not None:
        try:
            display(obj)
        except NameError:
            print(obj)


log(f"Using device: {device}")
log(f"Run directory: {RUN_DIR.resolve()}")
log(f"LR: {LR:.2e} (sqrt-scaled from {BASE_LR:.0e} @ bs={BASE_BATCH_SIZE})")
log(f"Classes: {NUM_CLASSES} ({CLASS_NAMES})")
torch.manual_seed(SEED)

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────
DATA_DIR = Path("../data/guide_eval")
candidates = sorted(DATA_DIR.glob("run-*.csv"), key=lambda p: p.stat().st_mtime)
if not candidates:
    raise FileNotFoundError(f"No run-*.csv found in {DATA_DIR.resolve()}")
csv_file = candidates[-1]
log(f"Using: {csv_file}")

df = pl.read_csv(csv_file)
log(f"Full dataset shape: {df.shape}")

# Assign class labels: 0–10 for reached, 11 for unreached
df = df.with_columns(
    pl.when(pl.col("reached") & pl.col("iterations_to_reach").is_not_null())
    .then(pl.col("iterations_to_reach").cast(pl.Int64))
    .otherwise(pl.lit(11))
    .alias("label")
)
log("Class distribution:", df["label"].value_counts().sort("label"))

# Subsample
if len(df) > SAMPLE_N:
    df = df.sample(SAMPLE_N, seed=SEED)
log(f"Sampled: {len(df)}")

# Train/val split
n_val = max(1, int(len(df) * VAL_FRAC))
df = df.sample(fraction=1.0, seed=SEED)  # shuffle
val_df = df.head(n_val)
train_df = df.tail(len(df) - n_val)
log(f"Train: {len(train_df)}, Val: {len(val_df)}")
log("Label distribution (train):", train_df["label"].value_counts().sort("label"))

In [ ]:
# ── Tokenizer ──────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_input(goal: str, guide: str) -> str:
    """Format a (goal, guide) pair as a text prompt for the model."""
    return f"Goal: {goal}\nGuide: {guide}"

In [ ]:
# ── Dataset ─────────────────────────────────────────────────────────────
class GuideDataset(Dataset):
    def __init__(self, data: pl.DataFrame):
        self.goals = data["goal"].to_list()
        self.guides = data["guide"].to_list()
        self.labels = data["label"].cast(pl.Int64).to_list()

    def __len__(self):
        return len(self.goals)

    def __getitem__(self, idx):
        text = format_input(self.goals[idx], self.guides[idx])
        enc = tokenizer(
            text,
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=False,
            return_tensors="pt",
        )
        n_tokens = enc["input_ids"].size(1)
        assert n_tokens <= MAX_LENGTH, (
            f"Sample {idx} has {n_tokens} tokens, exceeds MAX_LENGTH={MAX_LENGTH}. "
            f"Text: {text[:100]}..."
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_ds = GuideDataset(train_df)
val_ds = GuideDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# Sanity check
sample = train_ds[0]
log(f"Input shape: {sample['input_ids'].shape}")
log(f"Label: {sample['label'].item()} ({CLASS_NAMES[sample['label'].item()]})")
log(f"Decoded: {tokenizer.decode(sample['input_ids'], skip_special_tokens=True)[:200]}")

In [ ]:
# ── Model: Qwen backbone + classification head ────────────────────────
class GuideRanker(nn.Module):
    def __init__(self, model_name: str, num_classes: int):
        super().__init__()
        self.backbone = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            dtype=torch.bfloat16,
        )
        hidden_size = self.backbone.config.hidden_size
        # Freeze most of the backbone, only train last 2 layers + head
        for param in self.backbone.parameters():
            param.requires_grad = False
        for layer in self.backbone.model.layers[-2:]:
            for param in layer.parameters():
                param.requires_grad = True

        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes),
        )
        self.head.float()

    def forward(self, input_ids, attention_mask):
        attention_mask = attention_mask.to(torch.long)
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        # Use last hidden state at the last non-pad token
        hidden = outputs.hidden_states[-1]  # (B, seq_len, hidden)
        seq_lengths = attention_mask.sum(dim=1) - 1  # (B,)
        batch_idx = torch.arange(hidden.size(0), device=hidden.device)
        pooled = hidden[batch_idx, seq_lengths]  # (B, hidden)
        return self.head(pooled.float())  # (B, num_classes)


model = GuideRanker(MODEL_NAME, NUM_CLASSES).to(device)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
log(f"Trainable: {n_trainable:,} / {n_total:,} ({100 * n_trainable / n_total:.1f}%)")

In [ ]:
# ── Training ───────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss()

train_losses = []
val_losses = []
val_accs = []

TB_DIR = RUN_DIR / "tb"
writer = SummaryWriter(log_dir=str(TB_DIR))


def evaluate() -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    n = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Evaluating", leave=False, unit="batch"):
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            logits = model(ids, mask)
            total_loss += loss_fn(logits, labels).item() * len(labels)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            n += len(labels)
    return total_loss / n, correct / n


def train_epoch(step: int) -> tuple[float, int]:
    model.train()
    epoch_loss = 0.0
    n_samples = 0

    step_bar = tqdm(train_loader, desc="Training", leave=False, unit="batch")
    for batch in step_bar:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()), 1.0
        )
        optimizer.step()

        step_loss = loss.item()
        epoch_loss += step_loss * len(labels)
        n_samples += len(labels)
        step += 1

        writer.add_scalar("train/step_loss", step_loss, step)
        step_bar.set_postfix(loss=f"{epoch_loss / n_samples:.4f}")

    return epoch_loss / n_samples, step


global_step = 0
epoch_bar = tqdm(range(EPOCHS), desc="Epoch", unit="epoch")
for epoch in epoch_bar:
    train_loss, global_step = train_epoch(global_step)
    val_loss, val_acc = evaluate()
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    scheduler.step()

    writer.add_scalar("train/epoch_loss", train_loss, epoch + 1)
    writer.add_scalar("val/epoch_loss", val_loss, epoch + 1)
    writer.add_scalar("val/accuracy", val_acc, epoch + 1)
    writer.add_scalar("train/lr", scheduler.get_last_lr()[0], epoch + 1)

    log(
        f"Epoch {epoch + 1}/{EPOCHS}: "
        f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
        f"val_acc={val_acc:.4f}, lr={scheduler.get_last_lr()[0]:.1e}"
    )

writer.flush()
log(f"TensorBoard logs at: {TB_DIR.resolve()}")
log(f"View with: tensorboard --logdir {TB_DIR}")

In [ ]:
# ── Training curves ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, train_losses, "o-", label="train CE")
ax1.plot(epochs_range, val_losses, "o-", label="val CE")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Training curves")
ax1.legend()

ax2.plot(epochs_range, val_accs, "o-", color="green", label="val accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Validation accuracy")
ax2.legend()

fig.tight_layout()
fig.savefig(RUN_DIR / "training_curves.png", dpi=150)
plt.show()
log("Saved training_curves.png")

In [ ]:
# ── Evaluation: classification metrics ─────────────────────────────────
model.eval()
all_logits = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating", unit="batch"):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        logits = model(ids, mask)
        all_logits.append(logits.cpu())
        all_labels.extend(batch["label"].numpy())

all_logits = torch.cat(all_logits, dim=0)
all_probs = torch.softmax(all_logits, dim=1).numpy()
all_preds = all_logits.argmax(dim=1).numpy()
all_labels = np.array(all_labels)

present_classes = sorted(set(all_labels) | set(all_preds))
present_names = [CLASS_NAMES[i] for i in present_classes]

accuracy = (all_preds == all_labels).mean()
rho, rho_p = spearmanr(all_preds, all_labels)
macro_f1 = fbeta_score(
    all_labels,
    all_preds,
    beta=1,
    labels=present_classes,
    average="macro",
    zero_division=0,
)
weighted_f1 = fbeta_score(
    all_labels,
    all_preds,
    beta=1,
    labels=present_classes,
    average="weighted",
    zero_division=0,
)
macro_f2 = fbeta_score(
    all_labels,
    all_preds,
    beta=2,
    labels=present_classes,
    average="macro",
    zero_division=0,
)
weighted_f2 = fbeta_score(
    all_labels,
    all_preds,
    beta=2,
    labels=present_classes,
    average="weighted",
    zero_division=0,
)
kappa = cohen_kappa_score(all_labels, all_preds)
mcc = matthews_corrcoef(all_labels, all_preds)

log(f"Val Accuracy:   {accuracy:.4f}")
log(f"Macro F1:       {macro_f1:.4f}")
log(f"Weighted F1:    {weighted_f1:.4f}")
log(f"Macro F2:       {macro_f2:.4f}")
log(f"Weighted F2:    {weighted_f2:.4f}")
log(f"Cohen's κ:      {kappa:.4f}")
log(f"MCC:            {mcc:.4f}")
log(f"Spearman ρ:     {rho:.4f} (p={rho_p:.2e})")

report = classification_report(
    all_labels,
    all_preds,
    labels=present_classes,
    target_names=present_names,
    zero_division=0,
)
log("Classification report:")
print(report)

# Log final eval metrics to TensorBoard
writer.add_scalar("val/final_accuracy", accuracy, EPOCHS)
writer.add_scalar("val/final_macro_f1", macro_f1, EPOCHS)
writer.add_scalar("val/final_weighted_f1", weighted_f1, EPOCHS)
writer.add_scalar("val/final_macro_f2", macro_f2, EPOCHS)
writer.add_scalar("val/final_weighted_f2", weighted_f2, EPOCHS)
writer.add_scalar("val/final_cohen_kappa", kappa, EPOCHS)
writer.add_scalar("val/final_mcc", mcc, EPOCHS)
writer.add_scalar("val/final_spearman", rho, EPOCHS)
writer.close()

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds, labels=present_classes)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_xticks(range(len(present_names)))
ax.set_yticks(range(len(present_names)))
ax.set_xticklabels(present_names, rotation=45, ha="right")
ax.set_yticklabels(present_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix (accuracy={accuracy:.3f}, Spearman ρ={rho:.3f})")
fig.colorbar(im, ax=ax)

# Annotate cells with counts
for i in range(len(present_names)):
    for j in range(len(present_names)):
        val = cm[i, j]
        if val > 0:
            color = "white" if val > cm.max() / 2 else "black"
            ax.text(j, i, str(val), ha="center", va="center", color=color, fontsize=8)

fig.tight_layout()
fig.savefig(RUN_DIR / "confusion_matrix.png", dpi=150)
plt.show()
log("Saved confusion_matrix.png")

In [ ]:
# ── Per-class F1 bar chart ─────────────────────────────────────────────
per_class_f1 = fbeta_score(
    all_labels, all_preds, beta=1, labels=present_classes, average=None, zero_division=0
)
support = np.array([np.sum(all_labels == c) for c in present_classes])

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(present_names, per_class_f1, color="steelblue")
ax.set_xlabel("Class")
ax.set_ylabel("F1 Score")
ax.set_title("Per-class F1 (with sample counts)")
ax.set_ylim(0, 1.15)
for bar, f1, n in zip(bars, per_class_f1, support):  # pyright: ignore[reportArgumentType]
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{f1:.2f}\n(n={n})",
        ha="center",
        va="bottom",
        fontsize=9,
    )
fig.tight_layout()
fig.savefig(RUN_DIR / "per_class_f1.png", dpi=150)
plt.show()
log("Saved per_class_f1.png")

In [ ]:
# ── Confidence distribution: correct vs incorrect ─────────────────────
max_probs = all_probs.max(axis=1)
correct_mask = all_preds == all_labels

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(max_probs[correct_mask], bins=50, alpha=0.7, label="correct", density=True)
ax.hist(max_probs[~correct_mask], bins=50, alpha=0.7, label="incorrect", density=True)
ax.set_xlabel("Max predicted probability")
ax.set_ylabel("Density")
ax.set_title("Prediction confidence: correct vs incorrect")
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "confidence_distribution.png", dpi=150)
plt.show()
log("Saved confidence_distribution.png")

In [ ]:
# ── Off-by-one and top-k accuracy ─────────────────────────────────────
off_by_one = np.mean(np.abs(all_preds - all_labels) <= 1)
off_by_two = np.mean(np.abs(all_preds - all_labels) <= 2)

# Top-k accuracy from saved probabilities
top3_preds = np.argsort(all_probs, axis=1)[:, -3:]
top3_acc = np.mean([all_labels[i] in top3_preds[i] for i in range(len(all_labels))])
top5_preds = np.argsort(all_probs, axis=1)[:, -5:]
top5_acc = np.mean([all_labels[i] in top5_preds[i] for i in range(len(all_labels))])

log(f"Exact accuracy:  {accuracy:.4f}")
log(f"Within ±1:       {off_by_one:.4f}")
log(f"Within ±2:       {off_by_two:.4f}")
log(f"Top-3 accuracy:  {top3_acc:.4f}")
log(f"Top-5 accuracy:  {top5_acc:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
metrics = ["Exact", "Within ±1", "Within ±2", "Top-3", "Top-5"]
values = [accuracy, off_by_one, off_by_two, top3_acc, top5_acc]
bars = ax.bar(
    metrics, values, color=["steelblue", "#5ba3cf", "#8fc1e3", "#e8913a", "#f0b672"]
)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy metrics")
ax.set_ylim(0, 1.05)
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{val:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
    )
fig.tight_layout()
fig.savefig(RUN_DIR / "accuracy_metrics.png", dpi=150)
plt.show()
log("Saved accuracy_metrics.png")

In [ ]:
# ── Calibration curve (reliability diagram) ────────────────────────────
# For each sample, get the predicted probability assigned to the true class
true_class_probs = all_probs[np.arange(len(all_labels)), all_labels]

n_bins = 15
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_centers = []
bin_accs = []
bin_counts = []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (true_class_probs >= lo) & (true_class_probs < hi)
    if mask.sum() > 0:
        bin_centers.append((lo + hi) / 2)
        bin_accs.append(correct_mask[mask].mean())
        bin_counts.append(mask.sum())

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(8, 6), gridspec_kw={"height_ratios": [3, 1]}, sharex=True
)

ax1.plot([0, 1], [0, 1], "k--", label="perfectly calibrated")
ax1.plot(bin_centers, bin_accs, "o-", color="steelblue", label="model")
ax1.set_ylabel("Fraction correct")
ax1.set_title("Calibration curve (reliability diagram)")
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2.bar(bin_centers, bin_counts, width=1 / n_bins * 0.8, color="steelblue", alpha=0.7)
ax2.set_xlabel("Predicted probability of true class")
ax2.set_ylabel("Count")

fig.tight_layout()
fig.savefig(RUN_DIR / "calibration_curve.png", dpi=150)
plt.show()
log("Saved calibration_curve.png")

In [ ]:
# ── Save model ─────────────────────────────────────────────────────────
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": {
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
            "num_classes": NUM_CLASSES,
            "class_names": CLASS_NAMES,
            "val_accuracy": float(accuracy),
            "val_spearman": float(rho),  # pyright: ignore[reportArgumentType]
            "sample_n": SAMPLE_N,
            "epochs": EPOCHS,
        },
    },
    RUN_DIR / "ranker.pt",
)
log(f"Saved checkpoint to {RUN_DIR / 'ranker.pt'}")
log(f"\nAll outputs in: {RUN_DIR.resolve()}")